# Seed-variability comparison + WBF fusion

Run **after** `train_seed1/2/3.ipynb`. Reads the three `pred_seed*.json` + `metrics_seed*.json`, reports the seed **mean ± std** (the variance estimate for the report), then fuses the three seeds with **Weighted Box Fusion** and evaluates the ensemble — all on the TEST set.


In [ ]:
!pip -q install ensemble-boxes==1.0.9 pycocotools==2.0.7


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, glob, statistics
PROJECT = '/content/drive/MyDrive/arcade_multiseed'   # <-- same as training notebooks
OUT = f'{PROJECT}/outputs'
GT_COCO = f'{PROJECT}/data/test.json'                 # test COCO GT (stenosis)
STENOSIS_CAT = 26   # ARCADE COCO category id for 'stenosis'


## 1. Seed variability (mean ± std across the 3 seeds)


In [ ]:
members = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT}/metrics_seed*.json'))]
print('per-seed:', members)
for k in ['map50','map50_95','recall','precision','f1']:
    vals = [m[k] for m in members]
    sd = statistics.pstdev(vals) if len(vals)>1 else 0.0
    print(f'{k:9s}: {statistics.mean(vals):.4f} ± {sd:.4f}')


## 2. Weighted Box Fusion across the 3 seeds


In [ ]:
from pycocotools.coco import COCO
from ensemble_boxes import weighted_boxes_fusion
gt = COCO(GT_COCO)
# map file-stem -> (image_id, W, H)
info = {os.path.splitext(im['file_name'])[0]: (im['id'], im['width'], im['height'])
        for im in gt.loadImgs(gt.getImgIds())}
seed_preds = {s: json.load(open(f'{OUT}/pred_seed{s}.json')) for s in (1,2,3)}
# group each seed's detections by image stem
def by_img(plist):
    d = {}
    for p in plist: d.setdefault(p['image_id'], []).append(p)
    return d
grp = {s: by_img(seed_preds[s]) for s in (1,2,3)}

fused_dets = []
for stem,(img_id,W,H) in info.items():
    boxes_l, scores_l, labels_l = [], [], []
    for s in (1,2,3):
        bxs = grp[s].get(stem, [])
        bl = [[p['bbox'][0]/W, p['bbox'][1]/H, (p['bbox'][0]+p['bbox'][2])/W, (p['bbox'][1]+p['bbox'][3])/H] for p in bxs]
        boxes_l.append(bl or [[0,0,0,0]]); scores_l.append([p['score'] for p in bxs] or [0.0]); labels_l.append([0]*len(bxs) or [0])
    fb, fs, _ = weighted_boxes_fusion(boxes_l, scores_l, labels_l, iou_thr=0.55, skip_box_thr=0.001)
    for (x1,y1,x2,y2),sc in zip(fb, fs):
        fused_dets.append({'image_id': img_id, 'category_id': STENOSIS_CAT,
                           'bbox':[x1*W, y1*H, (x2-x1)*W, (y2-y1)*H], 'score': float(sc)})
json.dump(fused_dets, open(f'{OUT}/pred_fused_wbf.json','w'))
print('fused detections:', len(fused_dets))


## 3. Evaluate the fused ensemble (COCOeval, stenosis only)


In [ ]:
from pycocotools.cocoeval import COCOeval
dt = gt.loadRes(f'{OUT}/pred_fused_wbf.json')
ev = COCOeval(gt, dt, 'bbox'); ev.params.catIds=[STENOSIS_CAT]
ev.evaluate(); ev.accumulate(); ev.summarize()
print('FUSED mAP@50:95 =', round(float(ev.stats[0]),4), ' mAP@50 =', round(float(ev.stats[1]),4))


Compare **FUSED mAP** vs the single-seed **mean** from §1. A rise confirms the ensemble helps; the §1 ±std is your robustness/variance estimate for the report. `pred_seed*.json`, `metrics_seed*.json`, and `pred_fused_wbf.json` are saved in `outputs/` for the report and for further (seeds×scales) fusion.
